# 1 · The coffee network (LP)

**Act 1 of the workshop.** A specialty roaster buys green beans from 3 farms, roasts them in 2 roasteries and delivers roasted coffee to cafés in 4 cities. Find the cheapest plan.

| Stage | Places | Rule |
|---|---|---|
| Farms grow green beans | Colombia (CO), Ethiopia (ET), Brazil (BR) | sell at most their harvest |
| Roasteries roast them | Oakland (OAK), Brooklyn (BKN) | roast at most their capacity; roasting keeps 85% of the weight |
| Cafés sell the coffee | Chicago (CHI), Austin (AUS), Denver (DEN), Atlanta (ATL) | each city gets its demand |

Amounts are in tonnes (t). Costs are in **$1,000 per tonne**, so an objective of 1,604 means about $1.6 million.

Run the cells from top to bottom with **Shift + Enter**.

In [3]:
import pandas as pd
import plotly.graph_objects as go
import gamspy as gp

## Step 1 · The data

The data lives in ordinary pandas DataFrames. Nothing here is GAMSPy yet.

In [4]:
harvest_df  = pd.DataFrame({"f": ["CO", "ET", "BR"], "t": [150, 120, 200]})
capacity_df = pd.DataFrame({"r": ["OAK", "BKN"], "t": [260, 180]})
demand_df   = pd.DataFrame({"c": ["CHI", "AUS", "DEN", "ATL"], "t": [90, 60, 75, 80]})

# bean price + ocean freight, $1,000 per tonne of green beans
price_df = pd.DataFrame(
    [["CO", "OAK", 4.3], ["CO", "BKN", 4.4],
     ["ET", "OAK", 4.9], ["ET", "BKN", 4.7],
     ["BR", "OAK", 3.6], ["BR", "BKN", 3.4]],
    columns=["f", "r", "cost"])

# delivery to cafés, $1,000 per tonne of roasted coffee
deliver_df = pd.DataFrame(
    [["OAK", "CHI", 1.2], ["OAK", "AUS", 1.0], ["OAK", "DEN", 0.7], ["OAK", "ATL", 1.5],
     ["BKN", "CHI", 0.7], ["BKN", "AUS", 1.2], ["BKN", "DEN", 1.1], ["BKN", "ATL", 0.6]],
    columns=["r", "c", "cost"])
price_df

,f,r,cost
0,CO,OAK,4.3
1,CO,BKN,4.4
2,ET,OAK,4.9
3,ET,BKN,4.7
4,BR,OAK,3.6
5,BR,BKN,3.4


## Step 2 · Sets and parameters

A `Container` holds everything. **Sets** are the indices from the algebra (f, r, c). **Parameters** are the data, loaded straight from the DataFrames.

> **Gotcha:** GAMS symbol names ignore case, so a parameter called `"D"` (demand) and one called `"d"` (delivery) would clash. That's why the delivery cost is named `"deliver"` on the GAMS side, while we still call it `d` in Python.

In [5]:
m = gp.Container()

f = gp.Set(m, "f", description="farms", records=["CO", "ET", "BR"])
r = gp.Set(m, "r", description="roasteries", records=["OAK", "BKN"])
c = gp.Set(m, "c", description="cafe cities", records=["CHI", "AUS", "DEN", "ATL"])

H = gp.Parameter(m, "H", domain=f, description="harvest (t)", records=harvest_df)
K = gp.Parameter(m, "K", domain=r, description="roasting capacity (t)", records=capacity_df)
D = gp.Parameter(m, "D", domain=c, description="demand (t roasted)", records=demand_df)
p = gp.Parameter(m, "price", domain=[f, r], description="bean + freight cost", records=price_df)
d = gp.Parameter(m, "deliver", domain=[r, c], description="delivery cost", records=deliver_df)
eta = gp.Parameter(m, "eta", description="roasting yield", records=0.85)

## Step 3 · The model

Compare each line with the algebra slide:

- harvest: $\sum_r x_{f,r} \le H_f$
- capacity: $\sum_f x_{f,r} \le K_r$
- balance: $\sum_c y_{r,c} \le \eta \sum_f x_{f,r}$
- demand: $\sum_r y_{r,c} \ge D_c$

In [6]:
x = gp.Variable(m, "x", domain=[f, r], type="positive", description="t of green beans bought")
y = gp.Variable(m, "y", domain=[r, c], type="positive", description="t of roasted coffee shipped")

harvest  = gp.Equation(m, "harvest",  domain=f)
capacity = gp.Equation(m, "capacity", domain=r)
balance  = gp.Equation(m, "balance",  domain=r)
demand   = gp.Equation(m, "demand",   domain=c)

harvest[f]  = gp.Sum(r, x[f, r]) <= H[f]
capacity[r] = gp.Sum(f, x[f, r]) <= K[r]
balance[r]  = gp.Sum(c, y[r, c]) <= eta * gp.Sum(f, x[f, r])
demand[c]   = gp.Sum(r, y[r, c]) >= D[c]

cost = gp.Sum((f, r), p[f, r] * x[f, r]) + gp.Sum((r, c), d[r, c] * y[r, c])
coffee = gp.Model(m, "coffee", equations=m.getEquations(),
                  problem="LP", sense="min", objective=cost)

## Step 4 · Solve and read the plan

In [7]:
coffee.solve()
print(coffee.status)
print(f"Total cost: {coffee.objective_value:,.1f} ($1,000)")

ModelStatus.OptimalGlobal
Total cost: 1,604.2 ($1,000)


In [8]:
print("Bought, x (t green beans)")
display(x.pivot().round(1))
print("Shipped, y (t roasted coffee)")
display(y.pivot().round(1))

Bought, x (t green beans)


,OAK,BKN
CO,150.0,0.0
ET,8.8,0.0
BR,20.0,180.0


Shipped, y (t roasted coffee)


,CHI,AUS,DEN,ATL
OAK,17.0,60.0,75.0,0.0
BKN,73.0,0.0,0.0,80.0


**Questions to discuss**

1. Which farm feeds Brooklyn? Why that one?
2. Ethiopia sells only a little. Why?
3. Which roastery is full?

## Step 5 · Marginals are shadow prices

Every constraint comes back with a **marginal**: how much the total cost changes if you loosen that constraint by one unit. A constraint with spare room has marginal 0.

In [9]:
print("Roasting capacity")
display(capacity.records[["r", "level", "marginal"]])
print("Harvest")
display(harvest.records[["f", "level", "marginal"]])
print("Demand")
display(demand.records[["c", "level", "marginal"]])

Roasting capacity


,r,level,marginal
0,OAK,178.823529,0.000
1,BKN,180.000000,-0.625


Harvest


,f,level,marginal
0,CO,150.000000,-0.6
1,ET,8.823529,0.0
2,BR,200.000000,-1.3


Demand


,c,level,marginal
0,CHI,90.0,6.964706
1,AUS,60.0,6.764706
2,DEN,75.0,6.464706
3,ATL,80.0,6.864706


### Predict, then run

Write down a guess first: **if Brooklyn could roast 1 more tonne (181 t instead of 180 t), how much would we save?**

Then run the next cell.

In [11]:
base_cost = coffee.objective_value

K.setRecords([("OAK", 260), ("BKN", 181)])
coffee.solve()
print(f"{base_cost:,.3f}  ->  {coffee.objective_value:,.3f}")
print(f"Saved: {base_cost - coffee.objective_value:.3f} ($1,000)")

K.setRecords(capacity_df)   # put the data back
coffee.solve();

1,604.235  ->  1,603.610
Saved: 0.625 ($1,000)


## Step 6 · Models are experiments: a Brazilian drought

Once the model exists, a what-if is a for-loop. Cut Brazil's harvest in 5% steps, re-solve each time, and record the cost and the plan.

In [12]:
rows = []
for cut in [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    H.setRecords([("CO", 150), ("ET", 120), ("BR", 200 * (1 - cut))])
    coffee.solve()
    plan = x.pivot().reindex(index=["CO", "ET", "BR"], columns=["OAK", "BKN"]).fillna(0)
    rows.append({"cut %": round(100 * cut), "cost": coffee.objective_value,
                 "ET to BKN (t)": plan.loc["ET", "BKN"]})

H.setRecords(harvest_df)   # put the data back
drought = pd.DataFrame(rows)
drought["extra cost"] = drought["cost"] - drought["cost"].iloc[0]
drought.round(1)

,cut %,cost,ET to BKN (t),extra cost
0,0,1604.2,0.0,0.0
1,5,1617.2,0.0,13.0
2,10,1630.2,0.0,26.0
3,15,1643.2,10.0,39.0
4,20,1656.2,20.0,52.0
5,25,1669.2,30.0,65.0
6,30,1682.2,40.0,78.0
7,35,1695.2,50.0,91.0
8,40,1708.2,60.0,104.0
9,45,1721.2,70.0,117.0


In [16]:
colors = ["#F4961A" if t > 0 else "#9AA3AE" for t in drought["ET to BKN (t)"]]
fig = go.Figure(go.Bar(x=drought["cut %"], y=drought["extra cost"], width=3.5, marker_color=colors))
fig.update_layout(title="Orange: Brooklyn has to buy Ethiopian beans",
                  xaxis_title="Cut to Brazil's harvest (%)", yaxis_title="Extra cost ($1,000)",
                  template="simple_white", width=800, height=400)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

**Why is it a straight line of $13k per step?** Each 5% is 10 t, and the Brazil harvest marginal says each tonne is worth 1.3 ($1,000). The marginal predicted the experiment.

## Step 7 · Look under the hood

The equation listing shows the rows GAMSPy generated from your one line of code. Oakland's balance row is exactly the `-0.85` row of the matrix on the slides.

In [11]:
coffee.solve(options=gp.Options(equation_listing_limit=10))
print(balance.getEquationListing())

balance(OAK)..  - 0.85*x(CO,OAK) - 0.85*x(ET,OAK) - 0.85*x(BR,OAK) + y(OAK,CHI) + y(OAK,AUS) + y(OAK,DEN) + y(OAK,ATL) =L= 0 ; (LHS = 0)
balance(BKN)..  - 0.85*x(CO,BKN) - 0.85*x(ET,BKN) - 0.85*x(BR,BKN) + y(BKN,CHI) + y(BKN,AUS) + y(BKN,DEN) + y(BKN,ATL) =L= 0 ; (LHS = 0)


In [12]:
coffee.toLatex(path="coffee_latex", generate_pdf=False)   # the model as LaTeX math
coffee.toGams(path="coffee_gams")                          # the same model as a GAMS file

LaTeX (.tex) file has been generated under coffee_latex/coffee.tex
GAMS (.gms) file has been generated under coffee_gams/coffee.gms


## Your turn

**Core**

1. Oakland's roastery breaks down for a day and can only take 150 t. Change `K`, re-solve, and describe what changes in the plan.
2. Chicago's demand doubles to 180 t. Which constraint becomes the bottleneck? (Look at the marginals.)

**Stretch**

3. Add a fourth farm, Vietnam (VN): harvest 100 t, cost 3.9 to OAK and 4.2 to BKN. You only need to change data. Does the model code change at all?
4. A new rule: at least 20% of each roastery's beans must come from Ethiopia (a quality blend). Add this as a new `Equation` and see what it costs.

In [13]:
# Your work here
